# League baselines and optional warm starts

## Goal and setup
Build reusable feature columns from two pinned Premier League seasons. Compare
League/LOO and all three handoffs, then save/reuse transitioned rating states.
Use the `misc314_py314` kernel. Full explanations and input schemas are in the
[practical guide](../docs/analytics/league_warmup.md) and
[API reference](../docs/analytics/league_warmup_reference.md).

**Assumption:** no release column is supplied; earlier finished kickoff is a
retrospective proxy, not proof of when results became available. Warm-up remains opt-in.

In [1]:
from pathlib import Path
import pandas as pd
from xdiyo_analytics.data import load_seasons, select_stats
from xdiyo_analytics.histories import build_team_history
from xdiyo_analytics.features import (
    Stat, League, LeaveOneOut, RollingMean, RollingStd, WarmStart, SeededEMA,
    Hard, LinearFade, ObservationCount, Rating, evaluate_features,
)
from xdiyo_analytics.ratings import GlickoTransition, build_ratings, RatingRun

root = Path('C:/Users/luisi/Documents/Programming/Python/xDiyo')

In [2]:
data = load_seasons(
    root / 'data/xDiyo_data', ['23_24', '24_25'], leagues='Premier_League',
    tables=['matches', 'statistics', 'pregame'],
    record_dir=root / 'experiment/initial_population/selections',
)
history = build_team_history(select_stats(
    data, stats=[('ALL', 'Match overview', 'cornerKicks')],
))
corners = Stat('ALL', 'Match overview', 'cornerKicks')
print(f'{len(data.matches)} matches; {len(history)} team rows')

760 matches; 1520 team rows


## Steps: choose a population, then a policy
League defaults to a shared completed-round snapshot. LOO removes the focal team's
contributions **after** choosing the window, without refill. Std pools individual
observations. The three warm-up choices below affect only their wrapped expression.
Hard(1) hands off after one completed round; Fade and Count retain a decreasing EMA weight.

In [3]:
league = League(corners)
X = evaluate_features(history, {
    'team_mean': RollingMean(corners, 3),
    'league_mean': RollingMean(league, 3),
    'loo_mean': RollingMean(LeaveOneOut(league), 3),
    'league_std': RollingStd(league, 3),
})
assert X.index.equals(history.index)
print(X.shape)
X.tail(4)

(1520, 4)


,team_mean,league_mean,loo_mean,league_std
1516,2.666667,4.583333,4.684211,2.883569
1517,2.666667,4.583333,4.684211,2.883569
1518,7.333333,4.583333,4.438596,2.883569
1519,3.666667,4.583333,4.631579,2.883569


In [4]:
policies = {'hard': Hard(1), 'fade': LinearFade(1, 2), 'count': ObservationCount(5)}
warmed = evaluate_features(history, {
    name: WarmStart(RollingMean(corners, 3), SeededEMA(handoff=handoff))
    for name, handoff in policies.items()
})
previous_teams = set(history.loc[history.source_season.eq('23_24'), 'team_id'])
current = history.source_season.eq('24_25') & history.team_id.isin(previous_teams)
team = history.loc[current, 'team_id'].iloc[0]
view = current & history.team_id.eq(team)
pd.concat([history.loc[view, ['team_name', 'round']], warmed.loc[view]], axis=1).head(6)

,team_name,round,hard,fade,count
760,Fulham,1,5.578947,5.578947,5.578947
785,Fulham,2,4.333333,6.789474,6.380117
806,Fulham,3,6.333333,6.614035,6.734336
827,Fulham,4,7.000000,7.000000,6.654605
845,Fulham,5,5.333333,5.333333,4.994639
870,Fulham,6,5.000000,5.000000,5.180921


## Prepare and reuse rating transitions
This example inflates retained-team uncertainty by 20% at season entry. Location
and sigma stay unchanged at that transition. Movement shrinkage needs explicit
or evidence-derived movement context; newly observed teams are not automatically
promoted. The reference explains rating-only cohorts and custom adapters.

In [5]:
run = build_ratings(history, transition=GlickoTransition(phi_scale=1.2))
directory = root / 'experiment/league_warmup_demo/Premier_League_23_25/result_phi_1_2'
if not (directory / 'ratings.json').exists():
    run.save(directory)
saved = RatingRun.load(directory)
pd.testing.assert_frame_equal(run.snapshots, saved.snapshots)
rating_X = evaluate_features(history, {
    'rating': Rating('seasonal', fields=('rating', 'rd')),
}, ratings={'seasonal': saved})
assert rating_X.index.equals(history.index)
print(f'{len(saved.snapshots)} saved snapshots; feature shape {rating_X.shape}')
rating_X.loc[view].head(4)

1560 saved snapshots; feature shape (1520, 4)


,rating::result::team::rating,rating::result::team::rd,rating::result::opponent::rating,rating::result::opponent::rd
760,1464.390573,89.718097,1551.601013,89.040212
785,1448.110949,87.766249,1530.392883,253.570885
806,1468.064657,86.751210,1372.577606,249.381595
827,1464.538917,85.796272,1484.915416,84.905850


## Checks and next steps
The outputs preserve 1,520 team rows. The saved run has 1,560 snapshots, including
40 season-entry snapshots. Warm-up uses available history; it does not require a
full rolling window. These examples produce features and reusable state only.

See the [coverage and verification checklist](../docs/analytics/league_warmup_documentation_checklist.md)
for equations, all parameters, timing boundaries and independently checked behavior.